# BAA10Y Data Exploration

| Series | Why it's here |
|---|---|
| `BAA10Y` | Moody's Seasoned Baa Corporate Bond Yield Relative to Yield on 10-Year Treasury Constant Maturity  |

**Before running this notebook**, populate the local data cache:

```bash
uv run python scripts/fetch_fred.py
```

After that, no network calls are made — the notebook reads entirely from
the local cache, which is what the `CutoffEnforcer` discipline requires.

In [15]:
from datetime import datetime
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter
from aieng.forecasting.evaluation import ForecastingTask

from data import build_baa10y_target_service

## 1. Build the BAA10Y DataService

In [16]:
svc = build_baa10y_target_service(windows=(1, 5, 21))

## 2. Inspect the registered BAA10Y time series

In [17]:
svc.summary()

,series_id,description,source,units,frequency,n_obs,start,end
0,baa10y_change_1b,BAA10Y cumulative spread change over 1 busines...,"FRED (BAA10Y), derived",percentage-points,B,6911,2000-01-04,2026-06-30
1,baa10y_change_21b,BAA10Y cumulative spread change over 21 busine...,"FRED (BAA10Y), derived",percentage-points,B,6891,2000-02-01,2026-06-30
2,baa10y_change_5b,BAA10Y cumulative spread change over 5 busines...,"FRED (BAA10Y), derived",percentage-points,B,6907,2000-01-10,2026-06-30


## 3. Cutoff filtering

TODO

## 4. Plot the three BAA10Y time series

Code generated by Claude Sonnet 5.

In [18]:
as_of = "2027-1-1"
series_ids = ["baa10y_change_1b", "baa10y_change_5b", "baa10y_change_21b"]
series_colors = {
    "baa10y_change_1b": "#3987e5",   # blue (dark-mode step)
    "baa10y_change_5b": "#008300",   # green
    "baa10y_change_21b": "#d55181",  # magenta (dark-mode step)
}
series_labels = {
    "baa10y_change_1b": "1 business day",
    "baa10y_change_5b": "5 business days",
    "baa10y_change_21b": "21 business days",
}
recession_periods = [
    ("2001-03-01", "2001-11-01"),
    ("2007-12-01", "2009-06-01"),
    ("2020-02-01", "2020-04-01"),
]

# Each horizon's cumulative change lives on a very different scale (1b is tight
# around zero, 21b swings much wider), so they get their own y-axis per row
# rather than being forced onto one shared scale.
fig = make_subplots(
    rows=len(series_ids),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=[series_labels[sid] for sid in series_ids],
)

for row, series_id in enumerate(series_ids, start=1):
    s = svc.get_series(series_id, as_of=as_of).sort_values("timestamp")
    fig.add_trace(
        go.Scatter(
            x=s["timestamp"],
            y=s["value"],
            mode="lines",
            name=series_labels[series_id],
            line=dict(color=series_colors[series_id], width=1.6),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="Change (pp)", row=row, col=1)

for start, end in recession_periods:
    fig.add_vrect(
        x0=start,
        x1=end,
        fillcolor="#ffffff",
        opacity=0.14,
        line_width=0,
        row="all",
        col=1,
    )

fig.update_xaxes(title_text="Timestamp", row=len(series_ids), col=1)
fig.update_layout(
    title="BAA10Y cumulative spread change by horizon (independent y-scales, shaded: NBER recessions)",
    template="plotly_dark",
    height=700,
)
fig.show()

In [19]:
from sp500_forecasting import DEFAULT_COVARIATE_SERIES_IDS, build_sp500_multivariate_service

DATA_HISTORY_START = "2016-01-01"
REFRESH_CACHE = False

covariate_svc = build_sp500_multivariate_service(
    windows=(), #(1, 5, 21),
    include_covariates=True,
    covariate_series_ids=DEFAULT_COVARIATE_SERIES_IDS,
    start=DATA_HISTORY_START,
    refresh=REFRESH_CACHE,
)

covariate_svc.summary()

/home/coder/agentic-forecasting/implementations/sp500_forecasting/data.py:765: UserWarning: Skipping unavailable covariate 'gold_log_ret_1b_l1b': Could not fetch any configured gold FRED series (GOLDAMGBD228NLBM, GOLDPMGBD228NLBM). Check FRED availability/API key or override the gold covariate setup.
  _handle_covariate_error(SERIES_ID_GOLD_RETURN, exc)


,series_id,description,source,units,frequency,n_obs,start,end
0,cpi_mom_logdiff_l1b,"US CPI MoM log change, conservative release la...","FRED (CPIAUCSL), derived",log-change,B,2717,2016-01-15,2026-06-15
1,dollar_index_log_ret_1b_l1b,"Trade-weighted dollar index log return, lagged...","FRED (DTWEXBGS), derived",log-return,B,5347,2006-01-04,2026-07-02
2,fed_funds_level_l1b,"Effective federal funds rate, lagged 1 busines...",FRED (DFF),percent,B,18787,1954-07-02,2026-07-06
3,nasdaq_log_ret_1b_l1b,"NASDAQ Composite close-to-close log return, la...","Yahoo Finance (^IXIC), derived",log-return,B,2647,2016-01-06,2026-07-17
4,oil_log_ret_1b_l1b,"WTI oil spot log return, lagged 1 business day","FRED (DCOILWTICO), derived",log-return,B,10561,1986-01-06,2026-06-29
5,unemployment_rate_l1b,"US unemployment rate, conservative release lag...",FRED (UNRATE),percent,B,2739,2016-01-15,2026-07-15
6,ust10y_level_l1b,"US 10-year Treasury yield level, lagged 1 busi...",FRED (DGS10),percent,B,16829,1962-01-03,2026-07-06
7,ust2y10y_spread_l1b,"US 10Y minus 2Y Treasury spread, lagged 1 busi...","FRED (DGS10, DGS2), derived",percent-points,B,13069,1976-06-02,2026-07-06
8,vix_level_l1b,"CBOE VIX close level, lagged 1 business day",Yahoo Finance (^VIX),index-level,B,2647,2016-01-05,2026-07-15
9,vix_log_ret_1b_l1b,"CBOE VIX close-to-close log return, lagged 1 b...","Yahoo Finance (^VIX), derived",log-return,B,2646,2016-01-06,2026-07-15


## 5. Plot the covariate time series

Code generated by Claude Sonnet 5.

In [21]:
covariate_ids = covariate_svc.series_ids
covariate_plot_start = "2016-01-01"

# Fixed categorical order (validated for CVD-safety), cycled since there are
# more covariates than hues in the base palette.
covariate_palette = [
    "#3987e5",  # blue
    "#008300",  # green
    "#d55181",  # magenta
    "#c98500",  # yellow
    "#199e70",  # aqua
    "#d95926",  # orange
    "#9085e9",  # violet
    "#e66767",  # red
]

# Each covariate lives on its own scale (percent, log-return, index level, ...),
# so — as with the BAA10Y horizons above — every series gets its own row and
# y-axis rather than being forced onto one shared scale.
fig = make_subplots(
    rows=len(covariate_ids),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    subplot_titles=[covariate_svc.get_metadata(sid).description for sid in covariate_ids],
)

covariate_ends = []
for row, series_id in enumerate(covariate_ids, start=1):
    meta = covariate_svc.get_metadata(series_id)
    s = covariate_svc.get_series(series_id, as_of=as_of).sort_values("timestamp")
    # Some covariates (fed funds, 10Y yield, ...) have decades of history that
    # predate the others, so clip everyone to the same start for a fair view.
    s = s[s["timestamp"] >= covariate_plot_start]
    if len(s) > 0:
        covariate_ends.append(s["timestamp"].max())
    fig.add_trace(
        go.Scatter(
            x=s["timestamp"],
            y=s["value"],
            mode="lines",
            name=series_id,
            line=dict(color=covariate_palette[(row - 1) % len(covariate_palette)], width=1.2),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text=meta.units, row=row, col=1)

for start, end in recession_periods:
    fig.add_vrect(
        x0=start,
        x1=end,
        fillcolor="#ffffff",
        opacity=0.14,
        line_width=0,
        row="all",
        col=1,
    )

# Shared x-axes are linked via `matches` to this bottom axis, so the range
# only needs to (and, in plotly, only *can*) be set here — setting it with
# row="all" is a silent no-op on matched axes.
covariate_plot_end = max(covariate_ends)
fig.update_xaxes(title_text="Timestamp", row=len(covariate_ids), col=1)
fig.update_xaxes(range=[covariate_plot_start, covariate_plot_end], row=len(covariate_ids), col=1)
fig.update_annotations(font_size=11)
fig.update_layout(
    title="Covariates by series (independent y-scales, shaded: NBER recessions)",
    template="plotly_dark",
    height=220 * len(covariate_ids),
)
fig.show()